Merging Clinical and Gene expression data

In [20]:
import time

start_time = time.time()

print("Uche, Gloria, and Joshua are very close, but Eniola is also interested in joining the clique.")

end_time = time.time()
execution_time = end_time - start_time
print(execution_time)

Uche, Gloria, and Joshua are very close, but Eniola is also interested in joining the clique.
0.0006837844848632812


In [4]:
import pandas as pd

In [17]:
import pandas as pd
import os

# ==============================
# STEP 1: Load Clinical Data
# ==============================
print("📊 Step 1: Loading clinical data...")

clinical_data_list = []

for file_name in os.listdir():
    if "clinical" in file_name and file_name.endswith(".txt"):
        print("➡️ Found clinical file:", file_name)
        try:
            df = pd.read_csv(file_name, sep="\t", comment="#")
            clinical_data_list.append(df)
        except Exception as e:
            print(f"⚠️ Error reading {file_name}: {e}")

if clinical_data_list:
    combined_clinical = pd.concat(clinical_data_list, axis=1)
    combined_clinical = combined_clinical.loc[:, ~combined_clinical.columns.duplicated()]
else:
    print("⚠️ No clinical files found.")
    combined_clinical = pd.DataFrame()

# ==============================
# STEP 2: Load RNA-Seq Data
# ==============================
print("🧬 Step 2: Loading RNA-Seq data...")

rna_data_list = []

for file_name in os.listdir():
    if "rna_seq" in file_name and file_name.endswith(".tsv"):
        print("➡️ Found RNA-Seq file:", file_name)
        try:
            # Load the file, skip metadata line
            rna_data = pd.read_csv(file_name, sep="\t", skiprows=1)

            # Use the file name to extract the sample ID
            sample_id = file_name.replace(".tsv", "").replace("rna_seq_", "")
            rna_data["sample_id"] = sample_id  # Add it to every row

            # Keep only relevant columns
            if "gene_id" in rna_data.columns and "unstranded" in rna_data.columns:
                rna_data = rna_data[["gene_id", "unstranded", "sample_id"]]
                rna_data_list.append(rna_data)
            else:
                print(f"⚠️ Required columns not found in {file_name}. Skipping.")

        except Exception as e:
            print(f"⚠️ Error reading {file_name}: {e}")


if rna_data_list:
    all_rna_data = pd.concat(rna_data_list)
    try:
        rna_table = all_rna_data.pivot(index="gene_id", columns="sample_id", values="unstranded")
        rna_table = rna_table.T.reset_index()
        print("✅ RNA-Seq data pivoted successfully.")
    except Exception as e:
        print(f"⚠️ Error pivoting RNA-Seq data: {e}")
        rna_table = pd.DataFrame()
else:
    print("⚠️ No RNA-Seq files found.")
    rna_table = pd.DataFrame()

# ==============================
# STEP 3: Merge Both Datasets
# ==============================
print("🔗 Step 3: Merging clinical and RNA-Seq data...")

if not combined_clinical.empty and not rna_table.empty:
    if "bcr_patient_barcode" in combined_clinical.columns and "sample_id" in rna_table.columns:
        merged_data = pd.merge(
            combined_clinical,
            rna_table,
            left_on="bcr_patient_barcode",
            right_on="sample_id",
            how="inner"
        )
        merged_data.to_csv("patient_data_merged.csv", index=False)
        print("✅ All done! Saved as 'patient_data_merged.csv'")
    else:
        print("❌ Merge failed — required columns missing.")
else:
    print("❌ Could not merge — one of the tables is empty.")


📊 Step 1: Loading clinical data...
➡️ Found clinical file: nationwidechildrens.org_clinical_drug_brca.txt
➡️ Found clinical file: nationwidechildrens.org_clinical_follow_up_v1.5_brca.txt
➡️ Found clinical file: nationwidechildrens.org_clinical_follow_up_v2.1_brca.txt
➡️ Found clinical file: nationwidechildrens.org_clinical_follow_up_v4.0_brca.txt
➡️ Found clinical file: nationwidechildrens.org_clinical_follow_up_v4.0_nte_brca.txt
➡️ Found clinical file: nationwidechildrens.org_clinical_nte_brca.txt
➡️ Found clinical file: nationwidechildrens.org_clinical_omf_v4.0_brca.txt
➡️ Found clinical file: nationwidechildrens.org_clinical_patient_brca.txt
➡️ Found clinical file: nationwidechildrens.org_clinical_radiation_brca.txt
🧬 Step 2: Loading RNA-Seq data...
➡️ Found RNA-Seq file: 5a277308-f1bc-47b3-a397-aeabe9db8dc6.rna_seq.augmented_star_gene_counts.tsv
➡️ Found RNA-Seq file: fb3713b9-fad5-4d66-b419-8f53530b14cd.rna_seq.augmented_star_gene_counts.tsv
✅ RNA-Seq data pivoted successfully.
🔗 